# IBM Multi-GNN reference run on Colab (Phase 4)

Runs IBM's code (github.com/IBM/Multi-GNN, Apache 2.0, pinned at commit `252b025`) **unchanged** on HI-Small with their best GIN setup (`--emlps --reverse_mp --ego --ports`), then writes its per-transaction probabilities so `src/` can score them with our metrics.

**Their protocol differs from ours:** their val and test graphs contain every later transaction (neighbour sampling never filters by time), their test runs from Sep 9 to the end (our test **plus** the tail), and they pick the best epoch by val F1 at a 0.5 cutoff.

**Before the first run:** upload `data/raw/HI-Small_Trans.csv` from the laptop to `MyDrive/aml/raw/` (about 450 MB). Runtime → Change runtime type → **T4 GPU**.

Outputs go to `MyDrive/aml/ibm/`: the formatted data, the best checkpoint (saved whenever val F1 improves, so it survives a disconnect), the training log and `preds_val.npz` / `preds_test.npz`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/aml'
IBM = f'{DRIVE}/ibm'
!mkdir -p {IBM}/data/Small_HI {IBM}/models {IBM}/logs && ls -la {DRIVE}/raw

Clone IBM's repo at the pinned commit and install what Colab lacks. Their `LinkNeighborLoader` needs `pyg-lib` or `torch-sparse`, which come as wheels matched to Colab's torch and CUDA versions. **One compatibility patch** to their code, applied right after the clone: newer PyG makes `BaseTransform.forward` abstract, so their `AddEgoIds` (which only defines `__call__`) can't be instantiated. Renaming that method to `forward` restores their behaviour: PyG's `__call__` now calls `forward` on a shallow copy and returns the result.

In [ ]:
%cd /content
!rm -rf Multi-GNN && git clone -q https://github.com/IBM/Multi-GNN.git && cd Multi-GNN && git checkout -q 252b0252afca109d1d216c411c59ff70753b25fc && git log --oneline -1
%cd /content/Multi-GNN
!sed -i 's/    def __call__(self, data: Union\[Data, HeteroData\]):/    def forward(self, data: Union[Data, HeteroData]):/' train_util.py && grep -n 'def forward' train_util.py
import torch
v = '.'.join(torch.__version__.split('+')[0].split('.')[:2]) + '.0'
cu = 'cu' + torch.version.cuda.replace('.', '')
print(torch.__version__, cu, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch the runtime to T4')
!pip install -q torch_geometric wandb munch
!pip install -q pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{v}+{cu}.html
import torch_geometric, importlib.util
print('pyg', torch_geometric.__version__, '| pyg_lib', bool(importlib.util.find_spec('pyg_lib')), '| torch_sparse', bool(importlib.util.find_spec('torch_sparse')))

Format the CSV. IBM's `format_kaggle_files.py` needs `datatable`, which has no build for Colab's Python 3.12, so this is a line-for-line pandas port. It keeps the same IDs, assigned in order of first appearance and interleaved per row exactly as their loop does (sender then receiver; receiving then payment currency), because currency and format codes are numeric edge features in their model. It also keeps the same timestamps (seconds since 10 s before the first day's midnight), the same columns and the same stable sort by time. `EdgeID` is the row's position in the raw CSV, equal to our `src_row`. Skipped if the file is already in Drive.

In [ ]:
import os, numpy as np, pandas as pd
OUT = f'{IBM}/data/Small_HI/formatted_transactions.csv'
if os.path.exists(OUT):
    print('already formatted:', OUT)
else:
    raw = pd.read_csv(f'{DRIVE}/raw/HI-Small_Trans.csv', dtype=str)   # duplicate 'Account' header -> 'Account', 'Account.1'
    ts = pd.to_datetime(raw['Timestamp'], format='%Y/%m/%d %H:%M')
    first = ts.iloc[0].normalize() - pd.Timedelta(seconds=10)
    pair_codes = lambda a, b: pd.factorize(np.column_stack([a, b]).ravel())[0].reshape(-1, 2)
    cur = pair_codes(raw['Receiving Currency'], raw['Payment Currency'])                        # cur1 = receiving, cur2 = payment
    acc = pair_codes(raw['From Bank'] + raw['Account'], raw['To Bank'] + raw['Account.1'])
    df = pd.DataFrame({'EdgeID': np.arange(len(raw)), 'from_id': acc[:, 0], 'to_id': acc[:, 1],
                       'Timestamp': (ts - first).dt.total_seconds().astype('int64'),
                       'Amount Sent': raw['Amount Paid'].astype(float).round(6), 'Sent Currency': cur[:, 1],
                       'Amount Received': raw['Amount Received'].astype(float).round(6), 'Received Currency': cur[:, 0],
                       'Payment Format': pd.factorize(raw['Payment Format'])[0], 'Is Laundering': raw['Is Laundering'].astype(int)})
    df = df.sort_values('Timestamp', kind='stable')
    assert len(df) == 5_078_345 and df['Is Laundering'].sum() == 5177 and df.EdgeID.is_unique
    df.to_csv(OUT, index=False)
    print(f'{len(df):,} rows, {acc.max() + 1:,} accounts, {cur.max() + 1} currencies -> {OUT}')
!head -3 {OUT}

In [ ]:
import json
json.dump({'paths': {'aml_data': f'{IBM}/data', 'model_to_load': f'{IBM}/models', 'model_to_save': f'{IBM}/models'}},
          open('data_config.json', 'w'), indent=2)
!cat data_config.json

Train with IBM's command for their best Small-HI GIN, plus `--testing` (turns off wandb, which would otherwise ask for a login) and `--save_model --unique_name` (keeps the best-val-F1 checkpoint as `checkpoint_True.tar` in Drive; `--unique_name` is a flag in their parser). Their defaults: 100 epochs with no early stop, batch 8,192, 100 + 100 sampled neighbours, settings from `model_settings.json`. Every epoch also evaluates val and test, which is their protocol.

**Watch the first epoch's time** in the log (next cell). If 100 epochs won't fit in a Colab session, stop it and set `N_EPOCHS` lower; the write-up will say so. The best checkpoint so far stays in Drive either way.

In [ ]:
import subprocess, time
N_EPOCHS = 100
CMD = f'python -u main.py --data Small_HI --model gin --emlps --reverse_mp --ego --ports --testing --save_model --unique_name --n_epochs {N_EPOCHS}'
print(time.strftime('%H:%M'), CMD, flush=True)
with open(f'{IBM}/logs/train.log', 'w') as log:
    rc = subprocess.run(CMD, shell=True, stdout=log, stderr=subprocess.STDOUT).returncode
print(time.strftime('%H:%M'), 'exit', rc)
!grep -v -i warn {IBM}/logs/train.log | tail -8

Progress at any time: epochs done, and the last val and test F1 (theirs, at a 0.5 cutoff).

In [ ]:
!echo "epochs done: $(grep -c 'Validation F1' {IBM}/logs/train.log)"; grep -E 'Validation F1|Test F1|split|samples' {IBM}/logs/train.log | tail -8; ls -la {IBM}/models

Dump per-transaction probabilities from the best checkpoint, for val and for their test. Their own `--inference` mode is broken at this commit (it reads `args.avg_tps`, which the parser lacks, and passes `precrec=` to an `evaluate_hetero` that doesn't take it). So this rebuilds the model exactly as their `train_gnn` does and runs their `evaluate_hetero` steps, keeping the class-1 softmax instead of the argmax. Seed edges the sampler drops get no prediction; the coverage is printed.

In [ ]:
import sys, json, numpy as np, pandas as pd, torch, wandb
from torch_geometric.nn import to_hetero
sys.argv = ['main.py', '--data', 'Small_HI', '--model', 'gin', '--emlps', '--reverse_mp', '--ego', '--ports', '--testing', '--unique_name']
from util import create_parser, set_seed
from data_loading import get_data
from train_util import AddEgoIds, extract_param, add_arange_ids, get_loaders
from training import get_model

args = create_parser().parse_args(sys.argv[1:]); set_seed(args.seed)
data_config = json.load(open('data_config.json'))
tr, va, te, tr_i, va_i, te_i = get_data(args, data_config)
wandb.init(mode='disabled', config={k: extract_param(k, args) for k in ['lr', 'n_hidden', 'n_gnn_layers', 'w_ce1', 'w_ce2', 'dropout', 'final_dropout']}
           | {'epochs': args.n_epochs, 'batch_size': args.batch_size, 'model': args.model, 'data': args.data, 'num_neighbors': args.num_neighs, 'loss': 'ce', 'n_heads': None})
config = wandb.config
add_arange_ids([tr, va, te])
tr_l, va_l, te_l = get_loaders(tr, va, te, tr_i, va_i, te_i, AddEgoIds(), args)
device = torch.device('cuda:0')
model = to_hetero(get_model(next(iter(tr_l)), config, args), te.metadata(), aggr='mean')
ck = torch.load(f"{data_config['paths']['model_to_save']}/checkpoint_True.tar", map_location=device, weights_only=False)
model.load_state_dict(ck['model_state_dict']); model.to(device).eval()
print('best checkpoint from epoch', ck['epoch'])
edge_id = pd.read_csv(f'{IBM}/data/Small_HI/formatted_transactions.csv', usecols=['EdgeID']).EdgeID.to_numpy()

# their evaluate_hetero, keeping probabilities and the seed edges' arange ids (= row positions in the formatted file)
def dump(loader, inds, name):
    ids, probs = [], []
    for batch in loader:
        e = batch['node', 'to', 'node']
        seeds = loader.data['node', 'to', 'node'].edge_attr.detach().cpu()[inds.detach().cpu()[e.input_id.detach().cpu()], 0]
        mask = torch.isin(e.edge_attr[:, 0].detach().cpu(), seeds)
        ids.append(e.edge_attr[mask, 0].detach().cpu().long())
        e.edge_attr = e.edge_attr[:, 1:]
        batch['node', 'rev_to', 'node'].edge_attr = batch['node', 'rev_to', 'node'].edge_attr[:, 1:]
        with torch.no_grad():
            batch.to(device)
            out = model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict)[('node', 'to', 'node')]
            probs.append(torch.softmax(out[mask.to(device)], dim=-1)[:, 1].cpu())
    ids, probs = torch.cat(ids).numpy(), torch.cat(probs).numpy()
    keep = np.unique(ids, return_index=True)[1]
    np.savez(f'{IBM}/preds_{name}.npz', src_row=edge_id[ids[keep]], prob=probs[keep])
    print(f'{name}: {len(keep):,} of {len(inds):,} seed edges predicted ({len(keep) / len(inds):.2%}) -> {IBM}/preds_{name}.npz')

dump(va_l, va_i, 'val')
dump(te_l, te_i, 'test')